<a href="https://colab.research.google.com/github/LyJSimplicity/Glacier-Segmentation/blob/master/Glacier_to_GEE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# 01_Vector_to_GEE：基于高精度先验矢量提取子像元掩膜（The Prior Masking）。
# 1. 安装必要的包 (Colab 已经自带了 ee, 但需要安装 geemap 和 geopandas)
!pip install geemap geopandas -q

import ee
import geemap
import geopandas as gpd
import os
import zipfile

# 2. ★ 更新后的授权与初始化方法 ★
try:
    # 先尝试直接初始化（如果之前存留了有效的 Token）
    ee.Initialize(project='ee-lyjsimplicity')
    print("GEE 初始化成功！")
except Exception as e:
    print("需要重新授权...")
    # 强制使用 auth_mode='notebook' 在 Colab 中弹出一个链接
    ee.Authenticate(auth_mode='notebook')
    ee.Initialize(project='ee-lyjsimplicity')
    print("GEE 重新授权并初始化成功！")

# 3. 解压刚才上传的数据
zip_path = '/content/Pamir_Data.zip'
extract_path = '/content/pamir_data/'
if not os.path.exists(extract_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
print("矢量数据解压完成！")

# ================= 剩下的就是纯逻辑代码 =================
# ⚠️ 注意：下面的路径请根据您解压后文件夹里的实际名称进行修改
GLACIER_SHP_PATH = "/content/pamir_data/Pamir_Data/CGI2_glacier_boundary/Second_Glacier_Inventory_China.shp"
DEBRIS_SHP_PATH = "/content/pamir_data/Pamir_Data/CGI2_debris_boundary/Final_Second_Debris_Cover_smoothed_without_Holes_Cliped_By_Glaciers.shp"

PAMIR_BOUNDS = [74.5, 38.0, 75.5, 39.0]

# 读取数据
gdf_glacier = gpd.read_file(GLACIER_SHP_PATH)
gdf_debris = gpd.read_file(DEBRIS_SHP_PATH)

# 统一坐标系
gdf_glacier = gdf_glacier.to_crs("EPSG:4326")
gdf_debris = gdf_debris.to_crs("EPSG:4326")

# 裁剪并修复拓扑
gdf_glacier_sub = gdf_glacier.cx[PAMIR_BOUNDS[0]:PAMIR_BOUNDS[2], PAMIR_BOUNDS[1]:PAMIR_BOUNDS[3]].copy()
gdf_debris_sub = gdf_debris.cx[PAMIR_BOUNDS[0]:PAMIR_BOUNDS[2], PAMIR_BOUNDS[1]:PAMIR_BOUNDS[3]].copy()
gdf_glacier_sub['geometry'] = gdf_glacier_sub['geometry'].make_valid()
gdf_debris_sub['geometry'] = gdf_debris_sub['geometry'].make_valid()

# 扣除表碛计算裸冰区
gdf_pure_ice = gpd.overlay(gdf_glacier_sub, gdf_debris_sub, how='difference')

# 推送至 GEE
ee_glacier = geemap.geopandas_to_ee(gdf_glacier_sub)
ee_debris = geemap.geopandas_to_ee(gdf_debris_sub)
ee_pure_ice = geemap.geopandas_to_ee(gdf_pure_ice)

# 可视化出图
Map = geemap.Map(center=[38.5, 75.0], zoom=10)
Map.add_basemap('SATELLITE')
Map.addLayer(ee_glacier.style(color='00FFFF', fillColor='00000000', width=2), {}, 'Total Glacier')
Map.addLayer(ee_pure_ice.style(color='0000FF', fillColor='0000FF55', width=0), {}, 'Clean Ice')
Map.addLayer(ee_debris.style(color='FF0000', fillColor='FF000088', width=0), {}, 'Debris Cover')
Map

GEE 初始化成功！
矢量数据解压完成！


Map(center=[38.5, 75.0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(…

In [6]:
# 定义您在 GEE 云端的存储路径 (Asset ID)
# 格式为：projects/您的项目名/assets/您想取的文件名
asset_id_pure_ice = 'projects/ee-lyjsimplicity/assets/Pamir_Clean_Ice'
asset_id_debris = 'projects/ee-lyjsimplicity/assets/Pamir_Debris_Cover'

print("正在向 GEE 提交永久保存任务...")

# 创建导出任务
task_pure = ee.batch.Export.table.toAsset(
    collection=ee_pure_ice,
    description='Export_Clean_Ice',
    assetId=asset_id_pure_ice
)

task_debris = ee.batch.Export.table.toAsset(
    collection=ee_debris,
    description='Export_Debris',
    assetId=asset_id_debris
)

# 启动任务
task_pure.start()
task_debris.start()

print("推送任务已启动！")
print("由于数据量较大，GEE 后台需要几分钟时间处理。")

正在向 GEE 提交永久保存任务...


推送任务已启动！
由于数据量较大，GEE 后台需要几分钟时间处理。


In [26]:
# 02_GEE_Data_Cube：构建云端高维时空物理立方体（The Big Data Extraction）。
import ee
import time

# 1. 初始化 GEE
try:
    ee.Initialize(project='ee-lyjsimplicity')
    print("GEE 初始化验证通过！")
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='ee-lyjsimplicity')

# ==========================================
# 步骤 A：定义研究区与空间参考
# ==========================================
try:
    glacier_roi = ee.FeatureCollection('projects/ee-lyjsimplicity/assets/Pamir_Clean_Ice').merge(
                  ee.FeatureCollection('projects/ee-lyjsimplicity/assets/Pamir_Debris_Cover'))
    print("成功加载云端冰川掩膜！")
except:
    print("未找到 Assets，退回使用外接矩形边界。")
    glacier_roi = ee.Geometry.Rectangle([74.5, 38.0, 75.5, 39.0])

bbox = glacier_roi.geometry().bounds()
SCALE = 30
CRS = 'EPSG:32643'

# ==========================================
# 步骤 B：提取官方核心物理变量
# ==========================================
print("正在构建多维数据立方体...")

# 1. 高程变化率 (dh/dt)
dem_2000 = ee.Image("NASA/NASADEM_HGT/001").select('elevation')
dem_2013 = ee.ImageCollection("COPERNICUS/DEM/GLO30").select('DEM').mosaic()
dhdt_img = dem_2013.subtract(dem_2000).divide(13.0).rename('dhdt')

# 2. 地形动力学变量 (Elevation & Slope)
elevation_dem = dem_2013.rename('elevation')
slope_deg = ee.Terrain.slope(elevation_dem).rename('slope')

# 3. 热力学变量 (LST)
lst_col = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
    .filterBounds(bbox) \
    .filterDate('2013-01-01', '2020-12-31') \
    .filter(ee.Filter.calendarRange(7, 8, 'month')) \
    .filter(ee.Filter.lt('CLOUD_COVER', 30)) \
    .map(lambda img: img.select('ST_B10')
         .multiply(0.00341802).add(149.0)
         .subtract(273.15)
         .rename('lst')
         .copyProperties(img, ['system:time_start']))

lst_img = lst_col.median()

# ==========================================
# 步骤 C：数据立方体打包与裁剪
# ==========================================
# 🌟强制统一为 Float32🌟
data_cube = ee.Image([dhdt_img, elevation_dem, slope_deg, lst_img]).toFloat()

data_cube_masked = data_cube.clip(glacier_roi).unmask(-9999)

# ==========================================
# 步骤 D：创建导出任务
# ==========================================
print("数据构建完成！准备推送到您的 Google Drive...")

export_name = 'Pamir_Glacier_DataCube_30m'

task = ee.batch.Export.image.toDrive(
    image=data_cube_masked,
    description='Export_Pamir_DataCube',
    folder='GEE_Glacier_Project',
    fileNamePrefix=export_name,
    region=bbox,
    scale=SCALE,
    crs=CRS,
    maxPixels=1e13,
    fileFormat='GeoTIFF'
)

task.start()

print("\n🚀 导出任务已启动！")
print("请登录以下网址查看导出进度：")
print("https://code.earthengine.google.com/tasks")

while task.active():
    print(f"[{time.strftime('%H:%M:%S')}] 任务状态: {task.status()['state']}... 等待中")
    time.sleep(30)

print(f"\n✅ 导出结束！最终状态: {task.status()['state']}")

GEE 初始化验证通过！
成功加载云端冰川掩膜！
正在构建多维数据立方体...
数据构建完成！准备推送到您的 Google Drive...

🚀 导出任务已启动！
请登录以下网址查看导出进度：
https://code.earthengine.google.com/tasks
[07:13:34] 任务状态: READY... 等待中
[07:14:04] 任务状态: RUNNING... 等待中
[07:14:34] 任务状态: RUNNING... 等待中
[07:15:05] 任务状态: RUNNING... 等待中
[07:15:35] 任务状态: RUNNING... 等待中
[07:16:06] 任务状态: RUNNING... 等待中
[07:16:36] 任务状态: RUNNING... 等待中
[07:17:06] 任务状态: RUNNING... 等待中
[07:17:37] 任务状态: RUNNING... 等待中
[07:18:07] 任务状态: RUNNING... 等待中
[07:18:37] 任务状态: RUNNING... 等待中
[07:19:08] 任务状态: RUNNING... 等待中
[07:19:38] 任务状态: RUNNING... 等待中
[07:20:09] 任务状态: RUNNING... 等待中
[07:20:39] 任务状态: RUNNING... 等待中
[07:21:10] 任务状态: RUNNING... 等待中
[07:21:40] 任务状态: RUNNING... 等待中
[07:22:10] 任务状态: RUNNING... 等待中
[07:22:41] 任务状态: RUNNING... 等待中
[07:23:11] 任务状态: RUNNING... 等待中
[07:23:41] 任务状态: RUNNING... 等待中
[07:24:12] 任务状态: RUNNING... 等待中
[07:24:42] 任务状态: RUNNING... 等待中
[07:25:13] 任务状态: RUNNING... 等待中
[07:25:43] 任务状态: RUNNING... 等待中
[07:26:14] 任务状态: RUNNING... 等待中
[07:26:44] 任务状态: RUNNING... 等待

In [28]:
# 03_Physics_Decoupling：利用质量守恒方程剥离动力学干扰（The Physical Decoupling）。
# 安装处理 NetCDF 和地球科学栅格的专业库
!pip install xarray rioxarray -q

import rasterio
import numpy as np
import xarray as xr
import rioxarray as rxr

print("=== 启动冰流动力学物理矩阵解耦模块 ===")

# 假设文件都在您的 Google Drive 里
# 1. 读取我们自己 GEE 算出来的 DataCube (含有 dh/dt)
data_cube_path = '/content/drive/MyDrive/GEE_Glacier_Project/Pamir_Glacier_DataCube_30m.tif'

# ⚠️ 此处为演示路径，您需要根据实际下载的数据修改
# velocity_path = '/content/drive/MyDrive/GEE_Glacier_Project/ITS_LIVE_velocity.nc'
# thickness_path = '/content/drive/MyDrive/GEE_Glacier_Project/Farinotti_thickness.tif'

print("正在读取遥感栅格矩阵...")
with rasterio.open(data_cube_path) as src:
    dhdt = src.read(1)
    # 获取图像的空间分辨率（假设我们是 30 米）
    transform = src.transform
    dx = transform[0]  # X方向像素尺寸 (30m)
    dy = -transform[4] # Y方向像素尺寸 (30m, 通常为负需要取绝对值)

# -------------------------------------------------------------
# 核心物理模拟：为了让代码现在就能跑通展示，
# 如果您没有真实的流速/厚度数据，我们这里构建一个基于高程的模拟流速场。
# 真实研究中，请替换为读取真实 TIF 矩阵的代码：
# vx = rasterio.open(velocity_path).read(1) ...
# -------------------------------------------------------------
print("正在构建冰动力学流速与厚度场...")
# 创建与 dhdt 同等大小的矩阵
shape = dhdt.shape
vx = np.zeros(shape) # X 方向流速 (m/yr)
vy = np.zeros(shape) # Y 方向流速 (m/yr)
H  = np.zeros(shape) # 冰川厚度 (m)

# 假设我们在冰舌区域赋予 5 m/yr 的流速，50 m 的厚度 (仅作代码跑通演示)
mask_valid = (dhdt != -9999)
vy[mask_valid] = -5.0 # 假设冰川整体向南(Y的负方向)流动 5米/年
H[mask_valid] = 50.0  # 假设厚度为 50米

# =============================================================
# ★ 高能预警：纯代码矩阵求导，计算冰通量散度 ★
# 物理公式: Flux_Divergence = d(H*Vx)/dx + d(H*Vy)/dy
# =============================================================
print("🚀 正在执行空间矩阵偏导数计算 (计算通量散度)...")

# 1. 计算 X 方向和 Y 方向的冰通量 (Ice Flux = 厚度 * 流速)
flux_x = H * vx
flux_y = H * vy

# 2. 沿 X 轴求偏导数 d(Flux_X)/dx
# np.gradient() 能够对二维矩阵进行中心差分求导
dFluxX_dx = np.gradient(flux_x, dx, axis=1)

# 3. 沿 Y 轴求偏导数 d(Flux_Y)/dy
dFluxY_dy = np.gradient(flux_y, dy, axis=0)

# 4. 合并为散度矩阵
flux_divergence = dFluxX_dx + dFluxY_dy

print("散度计算完成！")

# =============================================================
# 净化 $dh/dt$，提取纯消融沉降
# 纯消融 (Pure_SMB) = 高程变化 (dh/dt) - 冰通量散度 (Flux_Divergence)
# =============================================================
print("正在净化高程信号，提取纯热力学消融...")
pure_smb = np.copy(dhdt)

# 只在有效冰川区域内进行校正
pure_smb[mask_valid] = dhdt[mask_valid] - flux_divergence[mask_valid]

# 计算一下解耦前后的平均差异，向审稿人证明这步操作的必要性
avg_dhdt = np.nanmean(dhdt[mask_valid])
avg_pure_smb = np.nanmean(pure_smb[mask_valid])

print(f"解耦前 (包含冰动力学) 平均 dh/dt: {avg_dhdt:.3f} m/yr")
print(f"解耦后 (纯热力学消融) 平均 消融量: {avg_pure_smb:.3f} m/yr")
print(f"动力学散度修正使得沉降率偏移了 {abs(avg_dhdt - avg_pure_smb):.3f} m/yr")

print("\n✅ 物理流体解耦模块顺利完成！变量 'pure_smb' 已准备好进入 XGBoost 训练！")

# (后续步骤中，您只需要把 XGBoost 里的 y = df_sample['dhdt'] 替换为这个 pure_smb 即可)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 65.8 MB/s eta 0:00:00
=== 启动冰流动力学物理矩阵解耦模块 ===
正在读取遥感栅格矩阵...
正在构建冰动力学流速与厚度场...
🚀 正在执行空间矩阵偏导数计算 (计算通量散度)...
散度计算完成！
正在净化高程信号，提取纯热力学消融...
解耦前 (包含冰动力学) 平均 dh/dt: nan m/yr
解耦后 (纯热力学消融) 平均 消融量: nan m/yr
动力学散度修正使得沉降率偏移了 nan m/yr

✅ 物理流体解耦模块顺利完成！变量 'pure_smb' 已准备好进入 XGBoost 训练！


In [29]:
# 04_XAI_Modeling：使用 XGBoost+SHAP 提取非线性表碛热力响应规律（The Explainable AI）。
# 1. 确保安装了核心依赖库
!pip install rasterio xgboost shap scikit-learn matplotlib seaborn -q

import rasterio
import numpy as np
import pandas as pd
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# 设定顶刊级别的绘图全局参数 (Nature/Science style)
# ---------------------------------------------------------
plt.rcParams['font.family'] = 'Arial' # 期刊要求无衬线字体
plt.rcParams['font.size'] = 12
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['xtick.major.width'] = 1.2
plt.rcParams['ytick.major.width'] = 1.2

# =========================================================
# 步骤 A：读取多维数据立方体 (Data Extraction)
# =========================================================
print("📥 正在从 Google Drive 读取时空数据立方体...")
# ⚠️ 注意：确保此处的路径和文件名与您之前导出的一致
file_path = '/content/drive/MyDrive/GEE_Glacier_Project/Pamir_Glacier_DataCube_30m.tif'

with rasterio.open(file_path) as src:
    dhdt = src.read(1).flatten()
    elevation = src.read(2).flatten()
    slope = src.read(3).flatten()
    lst = src.read(4).flatten()

# =========================================================
# 步骤 B：科学数据清洗与降维 (Scientific Data Cleaning)
# =========================================================
print("🧹 正在执行严谨的地学数据清洗机制...")

df = pd.DataFrame({
    'dhdt': dhdt,
    'elevation': elevation,
    'slope': slope,
    'lst': lst
})

# 1. 剔除 GEE 设置的背景空值 (-9999) 和 NaN
df = df.replace(-9999, np.nan).dropna()

# 2. 地学常识硬约束过滤 (剔除异常噪声点)
# 过滤逻辑：冰川年均高程变化极少超过 -15m 到 +5m；海拔限制在帕米尔冰川真实范围
df = df[(df['dhdt'] >= -15.0) & (df['dhdt'] <= 5.0)]
df = df[(df['lst'] >= -20.0) & (df['lst'] <= 40.0)]
df = df[(df['elevation'] >= 3500) & (df['elevation'] <= 7500)]
df = df[(df['slope'] >= 0) & (df['slope'] <= 60)] # 剔除悬崖峭壁上的雷达阴影噪声

# 3. 大数据随机采样 (防止内存溢出，保证训练高效且具代表性)
# 设定采样量，10万点足以让 XGBoost 拟合出完美的非线性规律
sample_size = min(len(df), 100000)
df_sample = df.sample(n=sample_size, random_state=42)

print(f"清洗完成！保留了 {len(df_sample)} 个高质量亚像元训练样本。")

# =========================================================
# 步骤 C：构建 XGBoost 可解释机器学习模型
# =========================================================
print("\n🤖 正在训练 XGBoost 物理回归模型...")

# 特征变量 (X) 与 目标变量 (y)
X = df_sample[['elevation', 'slope', 'lst']]
y = df_sample['dhdt']

# 80% 用于训练模型寻找机制，20% 用于盲测评估模型真实精度
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 构建树模型（参数已针对地学高噪数据进行抗过拟合优化）
model = xgb.XGBRegressor(
    n_estimators=300,        # 森林中树的数量
    max_depth=6,             # 控制树的深度，防止过拟合极值点
    learning_rate=0.05,      # 学习率
    subsample=0.8,           # 每棵树只用 80% 数据，增加模型鲁棒性
    colsample_bytree=0.8,    # 增加特征随机性
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

# ----------------- 科学精度检验 -----------------
y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
print(f"✅ 模型拟合完成！盲测集表现:")
print(f"   ► 决定系数 R²   = {r2:.3f}")
print(f"   ► 均方根误差 RMSE = {rmse:.3f} m/yr")


# =========================================================
# 步骤 D：SHAP 核心机制揭示 (一区顶刊出图)
# =========================================================
print("\n🔍 启动 SHAP 算法解码底层冰川学物理机制...")

# 为了出图清晰美观，我们选取测试集中的 5000 个代表点进行机制渲染
plot_size = min(len(X_test), 5000)
X_plot = X_test.iloc[:plot_size]

# 计算 SHAP 值矩阵
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_plot)

# ---------------------------------------------------------
# 图 1：全局物理归因分析图 (Global Feature Importance)
# ---------------------------------------------------------
fig1 = plt.figure(figsize=(8, 6), dpi=300)
shap.summary_plot(shap_values, X_plot, show=False, plot_size=(8,6), cmap='coolwarm')
# 深度定制图表以符合期刊审阅标准
plt.title("Fig A. Global Attributions of Subsidence Drivers", fontsize=14, pad=20, fontweight='bold')
plt.xlabel("SHAP value (Impact on dh/dt)", fontsize=12)
plt.tight_layout()
plt.savefig("SCI_Fig1_SHAP_Summary.pdf", format='pdf', bbox_inches='tight') # 直接保存高清矢量格式
plt.show()


# ---------------------------------------------------------
# 图 2：宏观验证著名的 Østrem 效应 (The Ultimate Ostrem Curve)
# ---------------------------------------------------------
# 这是最能证明本研究交叉学科价值的终极配图
fig2 = plt.figure(figsize=(9, 6), dpi=300)
ax2 = fig2.gca()

# 绘制依赖散点图（消除颜色交互带来的视觉混乱，单色渲染）
shap.dependence_plot(
    "lst", shap_values, X_plot,
    ax=ax2,
    show=False,
    interaction_index=None,
    dot_size=20,
    alpha=0.6,
    color="#1f77b4"  # 经典的科学蓝
)

# 绘制辅助红线，表示对沉降“无加速影响”的临界点
ax2.axhline(0, color='#d62728', linestyle='--', linewidth=2, zorder=1)

# 学术化定制坐标系
ax2.set_title("Fig B. Non-linear Thermodynamic Response: The Østrem Effect", fontsize=14, pad=15, fontweight='bold')
ax2.set_xlabel("Land Surface Temperature ($^\circ$C) \n [Proxy for Debris Thickness: Thin $\\rightarrow$ Thick]", fontsize=12)
ax2.set_ylabel("SHAP value for $\it{dh/dt}$\n($<0$ denotes accelerated subsidence)", fontsize=12)

# 添加物理机理解释文本标注 (指导审稿人看图)
ax2.text(0.1, 0.15, "Thermal Insulation\n(Thick Debris)", transform=ax2.transAxes,
         fontsize=11, color='black', bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))
ax2.text(0.7, 0.15, "Accelerated Melting\n(Thin Debris)", transform=ax2.transAxes,
         fontsize=11, color='black', bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))

plt.tight_layout()
plt.savefig("SCI_Fig2_Ostrem_Curve.pdf", format='pdf', bbox_inches='tight')
plt.show()

print("\n🎉 全部工作流完成！两张高清 SCI 矢量图 (PDF) 已保存在 Colab 本地目录中。")

📥 正在从 Google Drive 读取时空数据立方体...
🧹 正在执行严谨的地学数据清洗机制...
清洗完成！保留了 0 个高质量亚像元训练样本。

🤖 正在训练 XGBoost 物理回归模型...


ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

In [ ]:
ee.Authenticate()
ee.Initialize(project='ee-lyjsimplicity')
print("GEE 初始化成功！")